# Gmail Toolkit

This will help you getting started with the GMail [toolkit](/docs/concepts/tools/#toolkits). This toolkit interacts with the GMail API to read messages, draft and send messages, and more. For detailed documentation of all GmailToolkit features and configurations head to the [API reference](https://python.langchain.com/api_reference/google_community/gmail/langchain_google_community.gmail.toolkit.GmailToolkit.html).

## Setup

To use this toolkit, you will need to set up your credentials explained in the [Gmail API docs](https://developers.google.com/gmail/api/quickstart/python#authorize_credentials_for_a_desktop_application). Once you've downloaded the `credentials.json` file, you can start using the Gmail API.

### Installation

This toolkit lives in the `langchain-google-community` package. We'll need the `gmail` extra:

In [1]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY_S')

os.environ['MODEL'] = 'gemini/gemini-2.0-flash'

In [2]:
%pip install -qU langchain-google-community\[gmail\]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.2 MB/s eta 0:00:00


In [3]:
%pip install -Uq crewai crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.5/548.5 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.4/211.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 29.2 MB/s 

In [4]:
!pip install -Uq google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client


If you want to get automated tracing from runs of individual tools, you can also set your [LangSmith](https://docs.smith.langchain.com/) API key by uncommenting below:

In [ ]:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

## Instantiation

By default the toolkit reads the local `credentials.json` file. You can also manually provide a `Credentials` object.

In [6]:
from IPython import get_ipython
from IPython.display import display

from langchain_google_community import GmailToolkit

from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)

# Can review scopes here https://developers.google.com/gmail/api/auth/scopes
# For instance, readonly scope is 'https://www.googleapis.com/auth/gmail.readonly'
# **Changed**: Updated scopes to include valid Gmail API scopes.
# The scope 'https://mail.google.com/' is not a valid Gmail API scope
# and needs to be replaced with the appropriate scope for the desired access level.
# For read-only access, you can use 'https://www.googleapis.com/auth/gmail.readonly'.
# For read/write access (send emails), use 'https://www.googleapis.com/auth/gmail.modify'.
# See: https://developers.google.com/gmail/api/auth/scopes for a list of all available scopes.
# Define the necessary scopes for the Gmail API
scopes = ["https://www.googleapis.com/auth/gmail.modify"]  # Or 'https://www.googleapis.com/auth/gmail.readonly' for read-only access

# Request authorization and get credentials
# This will open a new tab in your browser, asking you to log in and authorize the application.
# After authorizing, copy the authorization code and paste it back into the notebook when prompted.
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

creds = None
if os.path.exists('token.json'):
    creds = Credentials.from_authorized_user_file('token.json', scopes)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            'credentials.json', scopes)
        creds = flow.run_local_server(port=0)
    with open('token.json', 'w') as token:
        token.write(creds.to_json())
# Build the Gmail API service
api_resource = build_resource_service(credentials=creds)  # Use creds to build the service

toolkit = GmailToolkit(api_resource=api_resource)

In [8]:
from langchain_google_community import GmailToolkit

from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)

# Can review scopes here https://developers.google.com/gmail/api/auth/scopes
# For instance, readonly scope is 'https://www.googleapis.com/auth/gmail.readonly'
# **Changed**: Updated scopes to include valid Gmail API scopes.
# The scope 'https://mail.google.com/' is not a valid Gmail API scope
# and needs to be replaced with the appropriate scope for the desired access level.
# For read-only access, you can use 'https://www.googleapis.com/auth/gmail.readonly'.
# For read/write access (send emails), use 'https://www.googleapis.com/auth/gmail.modify'.
# See: https://developers.google.com/gmail/api/auth/scopes for a list of all available scopes.
credentials = get_gmail_credentials(
    token_file="token.json",
    scopes=["https://www.googleapis.com/auth/gmail.modify"],  # Or 'https://www.googleapis.com/auth/gmail.modify' for send access
    client_secrets_file="credentials.json",
)
api_resource = build_resource_service(credentials=credentials)
toolkit = GmailToolkit(api_resource=api_resource)

In [12]:
tools1 = toolkit.get_tools()
tools1

[GmailCreateDraft(api_resource=<googleapiclient.discovery.Resource object at 0x7d4898ebc410>),
 GmailSendMessage(api_resource=<googleapiclient.discovery.Resource object at 0x7d4898ebc410>),
 GmailSearch(api_resource=<googleapiclient.discovery.Resource object at 0x7d4898ebc410>),
 GmailGetMessage(api_resource=<googleapiclient.discovery.Resource object at 0x7d4898ebc410>),
 GmailGetThread(api_resource=<googleapiclient.discovery.Resource object at 0x7d4898ebc410>)]

In [13]:
import nest_asyncio
nest_asyncio.apply()

In [14]:
from crewai import Agent, Task, Crew, LLM
from crewai.flow import Flow, start, listen

from crewai.project import agent, crew, task

llm1 = LLM(model="gemini/gemini-2.0-flash")

def manager() -> Agent:
    return Agent(
        role="Email Manager",
        goal="""You have to manager all email relevant tasks for user""",
        backstory="""You have to manager all email relevant tasks for user""",
        use_system_prompt=True,
        llm=llm1,
        tools=[tools1]
    )




def taks()-> Task:
  return Task(
      description="Task: {task}",
      expected_output="An answer to the question. and also share suggestion based on existing data. you can search any kind of thing after apply unfold text case.",
      agent=manager()
  )


def crew()-> Crew:
  return Crew(
      agents=[manager()],
      tasks=[taks()],
      verbose=True,

      planning=True,
      planning_llm=llm1
  )



class MyFlow(Flow):
  @start()
  def input_user(self):
    self.state['task'] = "show today emails"
    print("Step1:",self.state['task'])

  @listen(input_user)
  def my_crew(self):
    output =  crew().kickoff(inputs={"task": self.state['task']})
    return output.raw

my_flow = MyFlow()
result = my_flow.kickoff()



from IPython.display import Markdown
Markdown(result)
# what do you know about Alertli privacy?

 
[2025-03-12 09:51:15][🌊 FLOW CREATED: 'MYFLOW']: 2025-03-12 09:51:15.507618
 
[2025-03-12 09:51:15][🤖 FLOW STARTED: 'MYFLOW', 476DA746-BB26-4EE2-8350-ED8553B786DB]: 2025-03-12 09:51:15.513750
 Flow started with ID: 476da746-bb26-4ee2-8350-ed8553b786db
 
[2025-03-12 09:51:15][🤖 FLOW METHOD STARTED: 'INPUT_USER']: 2025-03-12 09:51:15.514443
Step1: show today emails
 
[2025-03-12 09:51:15][👍 FLOW METHOD FINISHED: 'INPUT_USER']: 2025-03-12 09:51:15.514649
 
[2025-03-12 09:51:15][🤖 FLOW METHOD STARTED: 'MY_CREW']: 2025-03-12 09:51:15.514932
 
[2025-03-12 09:51:15][❌ FLOW METHOD FAILED: 'MY_CREW']: 2025-03-12 09:51:15.515268
[Flow._execute_single_listener] Error in method my_crew: 1 validation error for Agent
tools.0
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=[GmailCreateDraft(api_res...ect at 0x7d4898ebc410>)], input_type=list]
    For further information visit https://errors.pydantic.dev/2.10/v/model_type
 
[2025-03-12 09:51:15][👍 FLOW FINI

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 1034, in _execute_single_listener
    listener_result = await self._execute_method(listener_name, method)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 876, in _execute_method
    raise e
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 846, in _execute_method
    else method(*args, **kwargs)
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-14-08d5d2c11b66>", line 49, in my_crew
    output =  crew().kickoff(inputs={"task": self.state['task']})
              ^^^^^^
  File "<ipython-input-14-08d5d2c11b66>", line 31, in crew
    agents=[manager()],
            ^^^^^^^^^
  File "<ipython-input-14-08d5d2c11b66>", line 9, in manager
    return Agent(
           ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pydantic/main.py", line 214, in __ini

<IPython.core.display.Markdown object>

In [75]:
from google.colab import userdata


from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

# Set up API keys
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()



In [80]:
from crewai import Agent, Task, Crew, LLM
from crewai.flow import Flow, start, listen

from crewai.project import agent, crew, task

llm1 = LLM(model="gemini/gemini-2.0-flash")

def manager() -> Agent:
    return Agent(
        role="My gmail Manager",
        goal="""You have to manager all email relevant tasks as users requiredments""",
        backstory="""you are responsible to date relvant thing first then find emails""",
        use_system_prompt=True,
        llm=llm1,
        tools=[search_tool,web_rag_tool,tools1],
    )




def taks()-> Task:
  return Task(
      description="Task: {task}",
      expected_output="An answer to the question. and also share suggestion based on existing data. you can search any kind of thing after apply unfold text case.",
      verbose=True,
      agent=manager()
  )


def crew()-> Crew:
  return Crew(
      agents=[manager()],
      tasks=[taks()],
      verbose=True,

      planning=True,
      planning_llm=llm1
  )



class MyFlow(Flow):
  @start()
  def input_user(self):
    self.state['task'] = input("Task: ")
    print("Step1:",self.state['task'])

  @listen(input_user)
  def my_crew(self):
    output =  crew().kickoff(inputs={"task": self.state['task']})
    return output.raw

my_flow = MyFlow()
result = my_flow.kickoff()



from IPython.display import Markdown
Markdown(result)
# what do you know about Alertli privacy?

 
[2025-03-12 03:22:23][🌊 FLOW CREATED: 'MYFLOW']: 2025-03-12 03:22:23.631012
 
[2025-03-12 03:22:23][🤖 FLOW STARTED: 'MYFLOW', 12B587C1-A348-4C0F-898D-3CC1953BEDF4]: 2025-03-12 03:22:23.633346
 Flow started with ID: 12b587c1-a348-4c0f-898d-3cc1953bedf4
 
[2025-03-12 03:22:23][🤖 FLOW METHOD STARTED: 'INPUT_USER']: 2025-03-12 03:22:23.633911
Task: share today date and emails
Step1: share today date and emails
 
[2025-03-12 03:22:32][👍 FLOW METHOD FINISHED: 'INPUT_USER']: 2025-03-12 03:22:32.096575
 
[2025-03-12 03:22:32][🤖 FLOW METHOD STARTED: 'MY_CREW']: 2025-03-12 03:22:32.096987
 
[2025-03-12 03:22:32][❌ FLOW METHOD FAILED: 'MY_CREW']: 2025-03-12 03:22:32.097337
[Flow._execute_single_listener] Error in method my_crew: 1 validation error for Agent
tools.2
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=[GmailCreateDraft(api_res...ect at 0x78a2a515c810>)], input_type=list]
    For further information visit https://errors.pydantic.dev/2.10/v/mo

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 1034, in _execute_single_listener
    listener_result = await self._execute_method(listener_name, method)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 876, in _execute_method
    raise e
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 846, in _execute_method
    else method(*args, **kwargs)
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-80-5530353289de>", line 50, in my_crew
    output =  crew().kickoff(inputs={"task": self.state['task']})
              ^^^^^^
  File "<ipython-input-80-5530353289de>", line 32, in crew
    agents=[manager()],
            ^^^^^^^^^
  File "<ipython-input-80-5530353289de>", line 9, in manager
    return Agent(
           ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pydantic/main.py", line 214, in __ini

<IPython.core.display.Markdown object>

In [82]:
# prompt: solve cell 80 error

from google.colab import userdata
import os
from langchain_google_community import GmailToolkit
from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)
import nest_asyncio
from crewai import Agent, Task, Crew, LLM
from crewai.flow import Flow, start, listen
from IPython.display import Markdown
from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

# ... (rest of your imports and environment setup)

# Corrected scopes for Gmail API access.  Choose the appropriate scope.
# 'https://www.googleapis.com/auth/gmail.readonly' for read-only access.
# 'https://www.googleapis.com/auth/gmail.modify' for read/write access (send emails).
credentials = get_gmail_credentials(
    token_file="token.json",
    scopes=["https://www.googleapis.com/auth/gmail.readonly"],  # Choose appropriate scope
    client_secrets_file="credentials.json",
)

llm1 = LLM(model="gemini/gemini-2.0-flash")

def manager() -> Agent:
    return Agent(
        role="My gmail Manager",
        goal="""You have to manager all email relevant tasks as users requiredments""",
        backstory="""you are responsible to date relvant thing first then find emails""",
        use_system_prompt=True,
        llm=llm1,
        tools=[search_tool,web_rag_tool,tools1],
    )




def taks()-> Task:
  return Task(
      description="Task: {task}",
      expected_output="An answer to the question. and also share suggestion based on existing data. you can search any kind of thing after apply unfold text case.",
      verbose=True,
      agent=manager()
  )


def crew()-> Crew:
  return Crew(
      agents=[manager()],
      tasks=[taks()],
      verbose=True,

      planning=True,
      planning_llm=llm1
  )



class MyFlow(Flow):
  @start()
  def input_user(self):
    self.state['task'] = input("Task: ")
    print("Step1:",self.state['task'])

  @listen(input_user)
  def my_crew(self):
    output =  crew().kickoff(inputs={"task": self.state['task']})
    return output.raw

my_flow = MyFlow()
result = my_flow.kickoff()



from IPython.display import Markdown
Markdown(result)
# what do you know about Alertli privacy?


 
[2025-03-12 03:25:52][🌊 FLOW CREATED: 'MYFLOW']: 2025-03-12 03:25:52.038932
 
[2025-03-12 03:25:52][🤖 FLOW STARTED: 'MYFLOW', 744E94CE-38A0-4E85-8A22-CD767A5FB852]: 2025-03-12 03:25:52.039763
 Flow started with ID: 744e94ce-38a0-4e85-8a22-cd767a5fb852
 
[2025-03-12 03:25:52][🤖 FLOW METHOD STARTED: 'INPUT_USER']: 2025-03-12 03:25:52.040191
Task: show todays emails
Step1: show todays emails
 
[2025-03-12 03:26:00][👍 FLOW METHOD FINISHED: 'INPUT_USER']: 2025-03-12 03:26:00.446168
 
[2025-03-12 03:26:00][🤖 FLOW METHOD STARTED: 'MY_CREW']: 2025-03-12 03:26:00.446578
 
[2025-03-12 03:26:00][❌ FLOW METHOD FAILED: 'MY_CREW']: 2025-03-12 03:26:00.446945
[Flow._execute_single_listener] Error in method my_crew: 1 validation error for Agent
tools.2
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=[GmailCreateDraft(api_res...ect at 0x78a2a515c810>)], input_type=list]
    For further information visit https://errors.pydantic.dev/2.10/v/model_type
 
[2025-0

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 1034, in _execute_single_listener
    listener_result = await self._execute_method(listener_name, method)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 876, in _execute_method
    raise e
  File "/usr/local/lib/python3.11/dist-packages/crewai/flow/flow.py", line 846, in _execute_method
    else method(*args, **kwargs)
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "<ipython-input-82-a7a6cdad8481>", line 76, in my_crew
    output =  crew().kickoff(inputs={"task": self.state['task']})
              ^^^^^^
  File "<ipython-input-82-a7a6cdad8481>", line 58, in crew
    agents=[manager()],
            ^^^^^^^^^
  File "<ipython-input-82-a7a6cdad8481>", line 35, in manager
    return Agent(
           ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pydantic/main.py", line 214, in __in

<IPython.core.display.Markdown object>

## API reference

For detailed documentation of all `GmailToolkit` features and configurations head to the [API reference](https://python.langchain.com/api_reference/community/agent_toolkits/langchain_community.agent_toolkits.gmail.toolkit.GmailToolkit.html).